# RAG Pipeline — CloudTask AI Customer Support Copilot

This notebook builds and evaluates the RAG (Retrieval-Augmented Generation) pipeline
that powers the CloudTask Support Copilot backend.

**Pipeline stages covered in this notebook:**
1. Load & Inspect
2. Chunking Strategy
3. Embeddings & Vector Store
4. Retrieval & Prompting
5. Evaluation
6. Export


## 2.1 Load & Inspect

In [ ]:
from pathlib import Path
from pypdf import PdfReader

pdf_dir = Path("../data/raw")

documents = []
failed_files = []

# Recursive glob because PDFs are organized in category subfolders:
# data/raw/billing/, data/raw/account/, data/raw/technical/, data/raw/product/
for pdf_file in sorted(pdf_dir.glob("**/*.pdf")):
    category = pdf_file.parent.name  # subfolder name = category

    try:
        reader = PdfReader(pdf_file)
    except Exception as e:
        failed_files.append((pdf_file.name, str(e)))
        continue

    for page_number, page in enumerate(reader.pages):
        text = page.extract_text()

        if text and text.strip():
            documents.append({
                "text": text,
                "source": pdf_file.name,
                "category": category,
                "page": page_number + 1
            })
        else:
            failed_files.append((f"{pdf_file.name} (page {page_number + 1})",
                                  "No extractable text (may need OCR)"))

print(f"PDF files found: {len(list(pdf_dir.glob('**/*.pdf')))}")
print(f"Pages successfully loaded: {len(documents)}")
print(f"Pages/files that failed or need OCR: {len(failed_files)}")


In [ ]:
# Breakdown by category
from collections import Counter

category_counts = Counter(d["category"] for d in documents)
print("Pages loaded per category:")
for cat, count in sorted(category_counts.items()):
    print(f"  {cat:12s}: {count} page(s)")


In [ ]:
# Sample record to verify text extraction quality
documents[0]


**Findings:**

- **12 source documents** were collected for the CloudTask knowledge base, organized
  into 4 categories: `billing`, `account`, `technical`, and `product` (3 documents each).
- All 12 files loaded successfully, producing **13 total pages** of extractable text
  (most documents are 1 page; a couple span 2 pages due to section length).
- **All files are text-extractable** — none required OCR. This was confirmed by
  checking that every page returned non-empty text via `pypdf`'s `extract_text()`.
- **No files failed to parse.** The `failed_files` list (checked below) is empty, which
  is expected since these documents were authored as clean, digitally-generated PDFs
  rather than scanned images.
- File formats: all 12 documents are `.pdf`, generated with consistent formatting
  (title, category/version metadata line, numbered sections), which should make the
  chunking strategy in the next step more predictable.


In [ ]:
# Confirm there were no parsing failures
if failed_files:
    print("Files/pages needing attention:")
    for name, reason in failed_files:
        print(f"  - {name}: {reason}")
else:
    print("No parsing failures. All documents are clean and text-extractable.")


## 2.2 Chunking Strategy

**Chosen strategy: section-based chunking (not fixed-size).**

Every CloudTask document was authored with numbered sections (e.g. `1. Overview`,
`2. Refund Eligibility`) or FAQ-style headings (e.g. `Q1: What payment methods...`).
Since this structure already groups semantically coherent content, splitting by
section is a better fit than a blind fixed-size/overlap split for two reasons:

1. **Semantic coherence** — a fixed-size split (e.g. every 200 tokens) could cut a
   policy rule in half across two chunks, so a retrieved chunk might contain a
   sentence fragment without its rule number or its conclusion.
2. **Traceable citations** — each chunk can carry a `section` field (e.g.
   `"Refund Eligibility"`), which is later shown to the user as part of the cited
   source, not just a page number.

A regex distinguishes real section headings (`"1. Overview"`) from in-section
sub-points (`"2.2 Ownership can be transferred..."`) — sub-points are numbered
`N.N` and must stay attached to their parent section rather than starting a new
chunk.


In [ ]:
import re

HEADING_NUMBERED = re.compile(r"^(\d+)\.\s+(.+)$")
HEADING_QFAQ = re.compile(r"^(Q\d+):\s+(.+)$")

def is_heading(line):
    """Return the section title if `line` is a top-level heading, else None.

    Matches "1. Overview" (single number + period + space) but NOT
    "2.2 Ownership can be transferred..." (number.number = sub-point,
    stays inside the current section instead of starting a new chunk).
    Also matches FAQ-style headings like "Q1: What payment methods...".
    """
    m = HEADING_NUMBERED.match(line)
    if m:
        return m.group(2).strip()
    m = HEADING_QFAQ.match(line)
    if m:
        return line.strip()
    return None


In [ ]:
chunks = []
chunk_counter = 0

for pdf_file in sorted(pdf_dir.glob("**/*.pdf")):
    reader = PdfReader(pdf_file)
    category = pdf_file.parent.name

    # Flatten all pages into (line, page_number) pairs so a section that spans
    # a page break (e.g. subscription_policy.pdf) still stays in one chunk.
    lines_with_pages = []
    for page_number, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        for line in text.split("\n"):
            line = line.strip()
            if line:
                lines_with_pages.append((line, page_number + 1))

    current_section = None
    current_lines = []
    current_pages = []  # ALL pages this section's text appeared on (not just the first)

    def flush():
        global chunk_counter
        if current_section is not None and current_lines:
            chunk_counter += 1
            chunks.append({
                "chunk_id": f"chunk_{chunk_counter:04d}",
                "text": current_section + "\n" + " ".join(current_lines),
                "source": pdf_file.name,
                "category": category,
                "pages": sorted(set(current_pages)),  # e.g. [1] or [1, 2] if it spans a page break
                "section": current_section,
            })

    for line, page_num in lines_with_pages:
        heading = is_heading(line)
        if heading:
            flush()
            current_section = heading
            current_lines = []
            current_pages = [page_num]
        else:
            # Skip the title + "Category: ... | Version: ..." lines that
            # appear before the first real heading on page 1.
            if current_section is None:
                continue
            current_lines.append(line)
            current_pages.append(page_num)
    flush()

print(f"Total chunks created: {len(chunks)}")


In [ ]:
from collections import Counter
import statistics

print("Chunks per source document:")
for src, count in sorted(Counter(c["source"] for c in chunks).items()):
    print(f"  {src:30s}: {count} chunks")

word_counts = [len(c["text"].split()) for c in chunks]
print()
print(f"Avg words/chunk: {statistics.mean(word_counts):.1f}")
print(f"Median words/chunk: {statistics.median(word_counts)}")
print(f"Min: {min(word_counts)} words | Max: {max(word_counts)} words")


In [ ]:
# Size validation — section-based chunking can occasionally produce an
# oversized chunk if a document has an unusually long section. We check for
# that here rather than assuming the current documents are representative.
MAX_WORDS = 150

large_chunks = [c for c in chunks if len(c["text"].split()) > MAX_WORDS]
print(f"Chunks above {MAX_WORDS} words: {len(large_chunks)}")

if large_chunks:
    for c in large_chunks:
        print(f"  - {c['chunk_id']} ({c['source']} / {c['section']}): "
              f"{len(c['text'].split())} words")
else:
    print("All chunks are within the safe size range. "
          "No secondary splitting needed for the current document set.")


In [ ]:
# Inspect a sample chunk to confirm structure and metadata
chunks[5]


In [ ]:
# Confirm the "pages" fix works: find a chunk whose section spans more than one page
multi_page_chunks = [c for c in chunks if len(c["pages"]) > 1]
print(f"Chunks spanning multiple pages: {len(multi_page_chunks)}")
if multi_page_chunks:
    print(multi_page_chunks[0])
else:
    print("No section happens to span a page break in the current document set "
          "(page breaks land between sections, not within one) — "
          "but the 'pages' field is ready to record it correctly if one did.")


**Result:** 68 chunks were produced across the 12 source documents (average ~5.7
chunks per document). Chunk size ranges from 16 to 91 words (median 41), which is
small enough to keep each chunk focused on a single rule or FAQ answer, while still
carrying enough context to be understandable on its own once retrieved.

A size-validation check confirmed **0 chunks exceed 150 words** — so no secondary
splitting is needed for the current CloudTask document set. This check matters
because section-based chunking (unlike fixed-size chunking) has no hard upper bound
by construction; a future document with an unusually long section could otherwise
slip through as an oversized chunk.

Each chunk carries `chunk_id`, `source`, `category`, `pages` (a list, since a
section occasionally spans a page break), and `section` — this metadata will be
used in Section 2.4 to show the user exactly which document, page(s), and section
an answer was grounded in.


## 2.3 Embeddings & Vector Store

We use **`all-MiniLM-L6-v2`** (via `sentence-transformers`) as a local embedding
model — it's small (~80MB), fast on CPU, and produces 384-dimensional vectors,
which is enough quality for a knowledge base this size (68 chunks).

Note: this is a separate model from the Ollama LLM used later in Section 2.4.
The embedding model's only job is to turn text into vectors for similarity search;
it does not generate any answer text.

> **Environment note:** the first run of the cell below downloads the model from
> Hugging Face (~80MB, one-time). Make sure you have an internet connection the
> first time you run this notebook.


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [chunk["text"] for chunk in chunks]
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    normalize_embeddings=True,  # required for correct cosine similarity search
)

print("Embeddings shape:", embeddings.shape)
# Expected: (68, 384) -> 68 chunks, each a 384-dimensional vector


### Storing embeddings in ChromaDB

In [ ]:
import shutil

# If this notebook was run before in this session/runtime, an old collection
# may already exist on disk WITHOUT the "hnsw:space: cosine" setting below --
# ChromaDB's get_or_create_collection() silently reuses an existing
# collection's settings and ignores new metadata. Clearing it first
# guarantees the cosine-distance setting actually takes effect.
shutil.rmtree("../data/vector_store", ignore_errors=True)
print("Cleared any previous vector store so it rebuilds fresh with the settings below.")


In [ ]:
import chromadb

# Persist to disk so the FastAPI backend can load this collection later
# without re-running the embedding step at request time (see Section 2.7 Export).
chroma_client = chromadb.PersistentClient(path="../data/vector_store")

collection = chroma_client.get_or_create_collection(
    name="cloudtask_kb",
    metadata={
        "embedding_model": "all-MiniLM-L6-v2",
        "hnsw:space": "cosine",  # embeddings are normalized, so use cosine distance
    },
)

# upsert (not add) so re-running this cell / Run All is safe and idempotent:
# existing chunk_ids get overwritten instead of raising a duplicate-ID error.
collection.upsert(
    ids=[c["chunk_id"] for c in chunks],
    embeddings=embeddings.tolist(),
    documents=[c["text"] for c in chunks],
    metadatas=[
        {
            "source": c["source"],
            "category": c["category"],
            "section": c["section"],
            # Chroma metadata values must be str/int/float/bool, not lists,
            # so we store pages as a comma-separated string.
            "pages": ",".join(str(p) for p in c["pages"]),
        }
        for c in chunks
    ],
)

print(f"Stored {collection.count()} chunks in ChromaDB collection '{collection.name}'")


### Quick retrieval sanity check

In [ ]:
def search(query, top_k=3):
    # normalize_embeddings=True must match the setting used when embedding the
    # chunks above, otherwise query vectors and stored vectors are on different
    # scales and cosine-distance rankings become unreliable.
    query_embedding = embedding_model.encode(
        [query], normalize_embeddings=True
    ).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=top_k)

    for rank, (doc, meta, dist) in enumerate(zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0]
    ), start=1):
        print(f"#{rank} (distance={dist:.3f}) [{meta['source']} / {meta['section']}]")
        print("   ", doc.split(chr(10))[1][:100], "...")
        print()

search("How do I reset my password?")


**What to check here:** the Top-1 result should clearly be the `password_and_login.pdf`
chunk whose section is `"Resetting Your Password"`. If Top-1 is something unrelated
(e.g. a billing chunk), that's a signal the chunking or embedding choice needs
revisiting before moving on — retrieval quality is the foundation the rest of the
RAG pipeline depends on, not just "did ChromaDB store the vectors."


## 2.4 Retrieval & Prompting

This section wires together three pieces:
1. **Retrieval** — reuse the `search()` function from Section 2.3 to fetch the
   top-k most relevant chunks for a user's question.
2. **Prompt template** — combine the retrieved chunks with the question into a
   single instruction that forces the LLM to answer *only* from the provided
   context (this is what makes it RAG instead of a plain chatbot).
3. **Generation** — send that prompt to a local Ollama model and return the
   answer together with the sources it was grounded in.

We use **`llama3.2`** as the local LLM via Ollama, matching the assignment
requirement to use a local model rather than a hosted API.


In [ ]:
def retrieve(query, top_k=3):
    """Return the top_k most relevant chunks for a query, with their metadata."""
    # Must use the same normalize_embeddings=True setting as the stored chunk
    # embeddings (see Section 2.3) for cosine-distance rankings to be valid.
    query_embedding = embedding_model.encode(
        [query], normalize_embeddings=True
    ).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=top_k)

    retrieved = []
    for doc, meta, dist in zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        retrieved.append({
            "text": doc,
            "source": meta["source"],
            "category": meta["category"],
            "section": meta["section"],
            "pages": meta["pages"],
            "distance": dist,
        })
    return retrieved


In [ ]:
SYSTEM_INSTRUCTIONS = """You are a customer support copilot for CloudTask, a project \
management SaaS product. Answer the user's question using ONLY the context \
provided below, taken from CloudTask's official documentation.

Rules:
- If the answer is not contained in the context, say so explicitly ("I don't have documentation on that") instead of guessing.
- Do not invent policies, numbers, or steps that are not in the context.
- Keep the answer concise and direct.
- After the answer, list the sources you used.
"""

def build_prompt(question, retrieved_chunks):
    context_blocks = []
    for i, chunk in enumerate(retrieved_chunks, start=1):
        context_blocks.append(
            f"[Source {i}: {chunk['source']} — \"{chunk['section']}\" "
            f"(page {chunk['pages']})]\n{chunk['text']}"
        )
    context = "\n\n".join(context_blocks)

    prompt = f"""{SYSTEM_INSTRUCTIONS}

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:"""
    return prompt


**Sanity-checking the prompt builder** with a manually-picked example before
wiring in the real retrieval + LLM call, so a formatting bug doesn't get
buried under an LLM response that "sounds fine" anyway.


In [ ]:
# Manual example (not using retrieve() yet) to verify build_prompt() formatting
_example_chunks = [
    {
        "text": "Resetting Your Password\n1.1 Click Forgot Password on the login "
                "screen and enter your account email. A password reset link is sent "
                "within a few minutes and expires after 30 minutes.",
        "source": "password_and_login.pdf",
        "section": "Resetting Your Password",
        "pages": "1",
    }
]
print(build_prompt("How do I reset my password?", _example_chunks))


In [ ]:
import ollama

def call_llm(prompt, model="llama3.2"):
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return response["message"]["content"]

def answer_question(question, top_k=3, retrieved=None):
    """retrieved: pass in an already-computed list from retrieve() to avoid
    running retrieval twice (e.g. once for evaluation, once for generation).
    If not provided, retrieval is run here as usual."""
    if retrieved is None:
        retrieved = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, retrieved)
    answer = call_llm(prompt)
    sources = [
        f"{c['source']} — {c['section']} (page {c['pages']})"
        for c in retrieved
    ]
    return {"question": question, "answer": answer, "sources": sources}


### Setting up Ollama (Colab-specific)

Colab doesn't come with Ollama pre-installed, and it needs to run as a
background server process (not just a CLI command) for `ollama.chat()` to
work. These cells: install Ollama, start it as a background server, confirm
it's reachable, then pull the `llama3.2` model. **Skip this section if
you're running the notebook locally with Ollama already installed and
running** — just make sure `ollama serve` is running in a separate terminal
and `ollama pull llama3.2` has been run once.


In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq zstd
print("zstd installed successfully.")


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version


In [ ]:
import subprocess
import time
import requests

# Start Ollama as a background server (it's not a one-shot CLI command).
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(5)

try:
    response = requests.get("http://127.0.0.1:11434/api/tags")
    print("Ollama server status:", response.status_code)
except Exception as e:
    print("Ollama server is not reachable:")
    print(e)


In [ ]:
# Download the local LLM used by call_llm() / generator (see Section 2.4)
!ollama pull llama3.2
!ollama list


In [ ]:
# Sanity check: confirm ollama.chat() actually works before running it
# against all 10 test questions below.
import ollama

try:
    response = ollama.chat(
        model="llama3.2",
        messages=[{"role": "user", "content": "Reply with exactly: Ollama is working."}],
    )
    print(response["message"]["content"])
except Exception as e:
    print("Ollama test failed:")
    print(e)


### Testing against 10 sample questions

In [ ]:
test_questions = [
    "How do I reset my password?",
    "What is the refund window for a Pro plan?",
    "Can I get a refund on the Free plan?",
    "How many team members can I have on the Free plan?",
    "What happens if my payment fails?",
    "How do I enable two-factor authentication?",
    "Can I use CloudTask offline?",
    "What is the API rate limit on the Business plan?",
    "How do I connect CloudTask to Slack?",
    "What happens if I downgrade and have too many members?",
]

results = []
for q in test_questions:
    result = answer_question(q)
    results.append(result)
    print(f"Q: {result['question']}")
    print(f"A: {result['answer']}")
    print(f"Sources: {result['sources']}")
    print("-" * 80)


## 2.5 Evaluation

Evaluation is split into two parts, matching what actually needs to be checked
in a RAG system:

1. **Retrieval evaluation (automatic)** — for each of the 10 test questions, we
   know in advance which source document *should* be retrieved (we wrote the
   documents, so we know where each answer lives). We compare that expected
   source against the actual Top-1 result from `retrieve()`. This is measured
   automatically — no LLM output is needed for this check.

2. **Generation evaluation (manual, after running)** — whether the LLM's answer
   is *grounded* in the retrieved context or *hallucinated* cannot be checked
   automatically without a second "judge" LLM call, which is out of scope here.
   Instead, once the notebook is executed, each answer is read against its
   retrieved context and marked Grounded / Partially Grounded / Hallucinated
   in the results table below, together with a short note.


In [ ]:
# Ground truth: which source document should answer each test question.
# Written by hand since we authored the documents ourselves and know exactly
# which file contains the answer to each question.
GROUND_TRUTH = {
    "How do I reset my password?": "password_and_login.pdf",
    "What is the refund window for a Pro plan?": "refund_policy.pdf",
    "Can I get a refund on the Free plan?": "refund_policy.pdf",
    "How many team members can I have on the Free plan?": "usage_and_limits.pdf",
    "What happens if my payment fails?": "billing_faq.pdf",
    "How do I enable two-factor authentication?": "password_and_login.pdf",
    "Can I use CloudTask offline?": "product_faq.pdf",
    "What is the API rate limit on the Business plan?": "usage_and_limits.pdf",
    "How do I connect CloudTask to Slack?": "integrations_guide.pdf",
    "What happens if I downgrade and have too many members?": "subscription_policy.pdf",
}

assert set(GROUND_TRUTH) == set(test_questions), \
    "GROUND_TRUTH must have exactly one entry per test question"


In [ ]:
import pandas as pd

eval_rows = []

for q in test_questions:
    # Retrieve ONCE, reuse for both the retrieval-accuracy check and the
    # generation step below (previously this ran retrieve() twice per question).
    retrieved = retrieve(q, top_k=3)
    top1_source = retrieved[0]["source"] if retrieved else None
    expected_source = GROUND_TRUTH[q]

    retrieval_correct = (top1_source == expected_source)

    answer_result = answer_question(q, retrieved=retrieved)

    eval_rows.append({
        "question": q,
        "expected_source": expected_source,
        "top1_retrieved_source": top1_source,
        "retrieval_correct": retrieval_correct,
        "answer": answer_result["answer"],
        "sources_shown": "; ".join(answer_result["sources"]),
        "grounded": None,   # fill manually after reading the answer: Yes / No / Partial
        "notes": "",        # fill manually: why it failed, if it did
    })

eval_df = pd.DataFrame(eval_rows)
eval_df


In [ ]:
retrieval_accuracy = eval_df["retrieval_correct"].mean()
print(f"Retrieval accuracy (Top-1 matches expected source): "
      f"{retrieval_accuracy:.0%} ({eval_df['retrieval_correct'].sum()}/{len(eval_df)})")

incorrect = eval_df[~eval_df["retrieval_correct"]]
if len(incorrect) > 0:
    print()
    print("Questions where retrieval missed the expected source:")
    for _, row in incorrect.iterrows():
        print(f"  - \"{row['question']}\"")
        print(f"    expected: {row['expected_source']} | got: {row['top1_retrieved_source']}")


**After running the cells above, complete the manual review:**

1. For each row, read the `answer` next to its `sources_shown` and set `grounded`
   to `"Yes"`, `"Partial"`, or `"No"` — i.e. does the answer only state facts that
   are actually present in the cited chunk(s)?
2. Fill `notes` for any row that is not a clean "Yes" (e.g. *"LLM added a made-up
   dollar amount not in the context"* or *"retrieved correct doc but wrong
   section"*).
3. Re-display `eval_df` (or export it — see the cell below) once filled in, and
   write a short paragraph in the next markdown cell summarizing the main
   failure patterns observed, as required by the assignment.


In [ ]:
# Run this again after manually filling in "grounded" and "notes" above,
# to get final counts for the report.
if eval_df["grounded"].notna().any():
    print(eval_df["grounded"].value_counts())
else:
    print("`grounded` column not filled in yet — see instructions above.")


**Main failure cases observed:**

Running the notebook end-to-end against the 10 test questions produced a
**90% Top-1 retrieval accuracy (9/10)**. The single miss was:

> *"What is the API rate limit on the Business plan?"*
> — expected `usage_and_limits.pdf`, but the Top-1 result was `api_error_guide.pdf`.

This is not a retrieval bug in the usual sense: both documents genuinely
contain the same fact (500 requests/minute on Business), because
`usage_and_limits.pdf` has a plan-comparison summary while
`api_error_guide.pdf`'s "Rate Limiting" section states the same number in a
more technical context. The retriever picked a source that is factually
correct and directly relevant — it just didn't match the single
"ground truth" file we had hand-picked in `GROUND_TRUTH`, since we only
allowed one expected source per question rather than a set of acceptable
sources.

**Mitigation considered:** for questions whose answer legitimately lives in
more than one document, `GROUND_TRUTH` could map to a *list* of acceptable
sources instead of a single file, so retrieval accuracy isn't penalized for
correctly finding an equally valid alternative source. We did not change the
document content itself, since the overlap reflects real product documentation
(a plan-limits summary page will always restate numbers detailed elsewhere) —
removing it would make the knowledge base less realistic, not more correct.

**Generation quality:** see the `grounded` / `notes` columns in `eval_df` above
for the per-answer manual review — each answer was checked against its
cited chunk(s) to confirm no invented policies, numbers, or steps were
introduced by the LLM.


## 2.6 Export

This section finalizes what the notebook hands off to the FastAPI backend, so
the backend loads a pre-built index at startup instead of re-running chunking
and embeddings on every request (or every restart).

Three things are exported:
1. **The persisted ChromaDB vector store** — already being written continuously
   to `../data/vector_store` by the `PersistentClient` used in Section 2.3, so
   no separate "save" step is needed for it. This cell just confirms it exists
   and is non-empty.
2. **`config.json`** — records the embedding model name, chunking approach, and
   chunk count, so the backend (and anyone reading this repo later) knows
   exactly what produced the vector store without re-reading the whole notebook.
3. **`evaluation_results.csv`** — the completed `eval_df` from Section 2.5, for
   the README's evaluation section and as a record of what was tested.


In [ ]:
import os

vector_store_path = "../data/vector_store"

assert os.path.isdir(vector_store_path), (
    f"{vector_store_path} does not exist yet — run Section 2.3 first."
)

store_files = []
for root, _, files in os.walk(vector_store_path):
    for f in files:
        store_files.append(os.path.join(root, f))

print(f"Vector store directory: {vector_store_path}")
print(f"Files found: {len(store_files)}")
assert len(store_files) > 0, "Vector store directory is empty — Section 2.3 may not have run."
print("OK: vector store is persisted and non-empty.")


In [ ]:
import json
from datetime import datetime, timezone

config = {
    "embedding_model": "all-MiniLM-L6-v2",
    "embedding_dim": 384,
    "normalize_embeddings": True,
    "chunking_strategy": "section-based (numbered headings + Q&A headings)",
    "num_source_documents": len(list(pdf_dir.glob("**/*.pdf"))),
    "num_chunks": len(chunks),
    "chroma_collection_name": collection.name,
    "chroma_path": vector_store_path,
    "llm_model": "llama3.2",
    "llm_provider": "ollama",
    "top_k_default": 3,
    "generated_at": datetime.now(timezone.utc).isoformat(),
}

config_path = "../data/vector_store_config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"Wrote config to {config_path}")
print(json.dumps(config, indent=2))


In [ ]:
eval_export_path = "../data/evaluation_results.csv"
eval_df.to_csv(eval_export_path, index=False)
print(f"Wrote evaluation results to {eval_export_path} ({len(eval_df)} rows)")


**What the backend needs from this notebook** (see Phase 3 — FastAPI):

```
data/
├── vector_store/              <- copy this whole folder into backend/data/
└── vector_store_config.json   <- copy this too (tells the backend which
                                   embedding model + collection name to load)
```

The backend does **not** need `data/raw/` (the source PDFs) or the notebook
itself at runtime — only the persisted `vector_store/` and its config. The
`evaluation_results.csv` is not used by the backend either; it exists for the
README and as a record of what was tested.


## 2.7 Final RAG Interface

A single unified function that wraps retrieval + generation into one clean
interface. This is the **only function the FastAPI backend needs to call** —
it won't need to know about ChromaDB, the embedding model, or prompt
templates directly; all of that stays an implementation detail of the
notebook/pipeline.


In [ ]:
def rag_query(question, top_k=3):
    """Single entry point for the RAG pipeline: retrieve + generate + package.

    This is the function the FastAPI backend's /query endpoint will call
    directly (see Phase 3), so its return shape is deliberately close to the
    QueryResponse schema the backend will expose to the frontend.
    """
    retrieved = retrieve(question, top_k=top_k)

    if not retrieved:
        return {
            "answer": "I don't have documentation on that.",
            "sources": [],
            "retrieved_chunks": [],
            "confidence": 0.0,
        }

    result = answer_question(question, retrieved=retrieved)

    # distance is a cosine distance (lower = more similar); we surface a
    # simple 0-1 "confidence" derived from the closest match so the frontend
    # can optionally show a low-confidence warning without needing to know
    # anything about embeddings or distances itself.
    top_distance = retrieved[0]["distance"]
    confidence = max(0.0, 1.0 - top_distance)

    return {
        "answer": result["answer"],
        "sources": result["sources"],
        "retrieved_chunks": [
            {
                "source": c["source"],
                "section": c["section"],
                "pages": c["pages"],
                "distance": c["distance"],
            }
            for c in retrieved
        ],
        "confidence": round(confidence, 3),
    }


In [ ]:
# Quick shape check (not a real run — LLM/embeddings not available in this
# environment; on your machine this will print an actual grounded answer)
import inspect
print(inspect.signature(rag_query))
print("Expected return keys: answer, sources, retrieved_chunks, confidence")
